# CS2235: Data Science Capstone Project
## Fintech: Cryptocurrency Risk Profiling via K-Means Clustering
**RV University | School of Computer Science & Engineering**  
*Programme:* B.Sc (Hons.) Data Science, Semester III  
*Instructors:* Prof. Muralidhar Billa, Prof. Thirukumaran Subbiramani  
*Student Lead:* Thanisha S Rao  

---

### Problem Statement
> **Grouping cryptocurrencies into similar risk categories based on how much their prices change (volatility, drawdowns) and how much they are traded (trading volume depth).**

---

### Alignment with Course Syllabus (Modules 1 to 5):
1. **Module 1 (Data Collection & Exploration):** Ingesting historical OHLCV data using Yahoo Finance API, working with Pandas & NumPy, understanding ratio and nominal data types.
2. **Module 2 (ETL Pipelines & Validation):** Extracting market data, transforming prices into percentage returns and volatility, loading clean CSVs, auditing missing values.
3. **Module 3 (Machine Learning Formulation):** Identifying this as an **Unsupervised Learning** problem because there are no predefined risk target labels.
4. **Module 4 (Feature Engineering, PCA & Clustering):** Standardizing features with `StandardScaler`, reducing dimensions with 2D PCA, finding optimal clusters using Elbow Method (Inertia/WCSS) and Silhouette Score, and training K-Means.
5. **Module 5 (Deployment & Financial Application):** Demonstrating real-world finance applications through an interactive Streamlit dashboard (`app.py`).


## 1. Environment Setup & Library Ingestion (Module 1)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

print("Dependencies successfully imported for CS2235 Capstone.")


## 2. Phase 2: Data Ingestion & Quality Checks (Modules 1 & 2)
We inspect the ingested daily historical OHLCV dataset covering 28 cryptocurrencies across 2 full years (731 daily records per asset).
- **Data Measurement Scale:** Price and Volume are **Ratio Scale** (continuous numerical with true zero). Symbol and Name are **Nominal Scale**.
- **Quality Checks:** Verification of zero missing values (`NaN = 0`) and timestamp alignment.

In [ ]:
# Load raw historical time-series data
raw_df = pd.read_csv('../data/raw/crypto_historical_raw.csv')
raw_df['Date'] = pd.to_datetime(raw_df['Date'])

print(f"Total time-series records: {len(raw_df):,}")
print(f"Total unique digital assets: {raw_df['Symbol'].nunique()}")
print(f"Time span: {raw_df['Date'].min().strftime('%Y-%m-%d')} to {raw_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Missing values count per column:\n{raw_df.isnull().sum()}")

raw_df.head()


## 3. Phase 3: ETL & Feature Engineering (Modules 2 & 4)
We transform the daily price time-series into cross-sectional risk and trading features:
1. **Annualized Volatility (%):** $\sigma_{ann} = \sigma_{daily} 	imes \sqrt{365} 	imes 100$ (measures daily price swings).
2. **Maximum Drawdown (MDD %):** The worst peak-to-trough drop percentage.
3. **Mean Daily Spread (%):** Average intraday price variation $rac{	ext{High} - 	ext{Low}}{	ext{Close}} 	imes 100$.
4. **Log10 Daily Volume (USD):** Log-transformed volume to compare high-volume stablecoins and smaller altcoins on a linear scale.
5. **Volatility-to-Volume Ratio:** An intuitive ratio measuring price fluctuation relative to trading volume depth.

In [ ]:
# Load processed quantitative risk features
features_df = pd.read_csv('../data/processed/crypto_risk_features.csv')
print(f"Extracted feature dataset shape: {features_df.shape}")

display_cols = ['Symbol', 'Name', 'Asset_Category', 'Annualized_Volatility_%', 
                'Max_Drawdown_%', 'Mean_Daily_Spread_%', 'Avg_Daily_Volume_USD', 'Log_Avg_Volume_USD']
features_df[display_cols].head(10)


### Feature Correlation Analysis (Module 4)

In [ ]:
feature_cols = [
    "Annualized_Volatility_%", "Max_Drawdown_%", "Mean_Daily_Spread_%", 
    "Log_Avg_Volume_USD", "Vol_to_Volume_Ratio"
]

plt.figure(figsize=(8, 6), dpi=150)
sns.heatmap(features_df[feature_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Engineered Features (CS2235 Module 4)", fontsize=12, fontweight="bold")
plt.show()


## 4. Phase 4: K-Means Clustering & Hyperparameter Tuning (Module 4)
Before clustering, we normalize features with **StandardScaler** ($Z = rac{X - \mu}{\sigma}$) so that volume does not dominate price percentages.
We evaluate optimal clusters ($k \in [2, 7]$) using:
1. **The Elbow Method (Inertia / WCSS):** Looking for the point where within-cluster variance stabilizes.
2. **Silhouette Score:** Measuring cluster separation and cohesion (higher is better).

In [ ]:
CLUSTER_FEATURES = [
    "Annualized_Volatility_%",
    "Max_Drawdown_%",
    "Mean_Daily_Spread_%",
    "Log_Avg_Volume_USD",
    "Vol_to_Volume_Ratio"
]

X = features_df[CLUSTER_FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Evaluate K across 2 to 7 (Hyperparameter tuning)
k_range = range(2, 8)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=150)
ax1.plot(list(k_range), inertias, marker='o', linewidth=2, color='#1f77b4')
ax1.axvline(4, color='red', linestyle='--', label='Elbow at k=4')
ax1.set_title("Elbow Method (Inertia vs. k)", fontweight="bold")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.legend()

ax2.plot(list(k_range), silhouettes, marker='s', linewidth=2, color='#2ca02c')
ax2.axvline(4, color='red', linestyle='--', label='Optimal k=4 (Peak = 0.3868)')
ax2.set_title("Silhouette Score vs. k (Module 4)", fontweight="bold")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.legend()
plt.show()


## 5. Phase 5: Dimensionality Reduction (PCA Basics) & Visualization (Modules 4 & 5)
We project our 5 features into 2 Principal Components using **PCA** (which explains **96.94%** of the total variance) to visually inspect our 4 risk clusters.

In [ ]:
# Load final clustered dataset
clustered_df = pd.read_csv('../data/processed/crypto_clustered_results.csv')

plt.figure(figsize=(10, 6), dpi=150)
tier_colors = {
    "Ultra-Low Risk": "#2ca02c",
    "Low-Moderate Risk": "#1f77b4",
    "Moderate-High Risk": "#ff7f0e",
    "Extreme Risk": "#d62728"
}

for tier, col in tier_colors.items():
    sub = clustered_df[clustered_df["Risk_Tier"] == tier]
    plt.scatter(sub["PCA1"], sub["PCA2"], label=tier, color=col, s=130, edgecolors="black", linewidths=0.8)

for _, row in clustered_df.iterrows():
    plt.annotate(row["Symbol"], (row["PCA1"], row["PCA2"]), xytext=(4, 4), textcoords="offset points", fontsize=8, fontweight="bold")

plt.title("2D Principal Component Analysis (PCA) Risk Clusters (96.94% Variance)", fontsize=12, fontweight="bold")
plt.xlabel("PCA Component 1 (Price Volatility & Drawdown)", fontweight="bold")
plt.ylabel("PCA Component 2 (Trading Volume Depth)", fontweight="bold")
plt.legend(title="Risk Category", frameon=True, facecolor="white")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 6. The Core Problem Statement Plot: Price Volatility vs. Trading Volume
We graph how much cryptocurrency prices change against how much they are traded.

In [ ]:
plt.figure(figsize=(10, 5.5), dpi=150)

for tier, col in tier_colors.items():
    sub = clustered_df[clustered_df["Risk_Tier"] == tier]
    plt.scatter(sub["Log_Avg_Volume_USD"], sub["Annualized_Volatility_%"], 
                label=tier, color=col, s=130, edgecolors="black", linewidths=0.8)

for _, row in clustered_df.iterrows():
    plt.annotate(row["Symbol"], (row["Log_Avg_Volume_USD"], row["Annualized_Volatility_%"]), 
                 xytext=(5, 4), textcoords="offset points", fontsize=8, fontweight="bold")

plt.title("Core Problem Statement: Volatility vs. Trading Volume Depth", fontsize=12, fontweight="bold")
plt.xlabel("Log10 Average Daily Trading Volume (USD)", fontweight="bold")
plt.ylabel("Annualized Price Volatility (%)", fontweight="bold")
plt.legend(title="Risk Tier", frameon=True, facecolor="white")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 7. Cluster Risk Profile Breakdown & Insights (Module 5)
Below is the empirical synthesis of the 4 discovered risk tiers:

In [ ]:
summary_table = clustered_df.groupby("Risk_Tier").agg({
    "Symbol": ["count", lambda s: ", ".join(s)],
    "Annualized_Volatility_%": "mean",
    "Max_Drawdown_%": "mean",
    "Avg_Daily_Volume_USD": lambda v: f"${v.mean()/1e6:.1f}M"
})
summary_table.columns = ["Coin Count", "Cryptocurrencies", "Mean Volatility (%)", "Mean MDD (%)", "Mean Daily Volume"]
summary_table


### Conclusions & Practical Takeaways:
1. **Tier 1 (Ultra-Low Risk / Pegged Stability):** USDT, USDC, DAI show near-zero volatility (<0.7%) and minimal drawdowns (<0.5%), serving as stable baseline cash equivalents.
2. **Tier 2 (Low-to-Moderate Risk / Blue-Chips):** BTC, ETH, BNB, SOL, XRP benefit from multi-billion dollar daily liquidity and manageable volatility (50% - 75%).
3. **Tier 3 (Moderate-to-High Risk / Utility Ecosystems):** ADA, AVAX, DOT, LINK, etc. exhibit active volume with 75% - 95% volatility, ideal for growth allocation.
4. **Tier 4 (Extreme Speculative Risk / Meme Tokens):** BONK, WIF, FLOKI, SHIB show extreme volatility (>100%) and severe drawdowns (>85%), requiring strict caution.
